
**Proje metni:** *GRPO'da aynı soru için üretilen doğru ve yanlış cevapların tokenlarının toplam top10 olasılıkları (en yüksek olasılıklı 10 tokenın olasılıklarının toplamı) (moving avg.) nasıl değişiyor? X ekseni token numarası, Y ekseni top10 toplamı. Bu grafikten özellikler çıkararak, cevabın doğru mu yanlış mı olduğunu tahmin eden bir ML algoritması geliştirip test edin. Qwen3.5 4B modelinin hem doğru hem yanlış cevaplar ürettiği gsm8k_tr sorularını belirleyin. Her soru için 10'ar adet cevap ürettirin. 250 eğitim sorusu (2500 cevap) ile eğitip, 250 test sorusu (2500 cevap) test edin.*


In [ ]:
# Kütüphaneler yükleniyor performansı artırmak için
!pip install -q transformers accelarate 

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessor
from google.colab import drive
import torch, torch.nn.functional as F, re, json, time, os, numpy as np
import random

# Drive üzerinde yer alan veriye erişim sağlanmakta, ardından verilerin kayıt edileceği klasör oluşturulmakta.
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/proje5/data', exist_ok=True)

In [ ]:
# soruların alıancağı dataset yükleniyor
ds = load_dataset("ytu-ce-cosmos/gsm8k_tr")

# dataset 3. ödevden bilindiği üzere latex formatı dönüşümlerinden, nokta virgül ifade kullanımlarından ve 
# bazen de son satırında yer alan belirsiz ifadelerden dolayı ön işleme tabii tutulacak.

# Cevapların son satırında sayı olup olmadığını kontrol etmek için tüm veri seti taranmakta.
# Bu analiz is_correct fonksiyonunun hangi edge case'leri ele alması gerektiğini belirlemek için yapılmaktadır.

def sayi_tipi_tespit(metin):
    """Cevap metninin son satırındaki içeriği sınıflandırır."""
    son = metin.strip().split('\n')[-1].lower()
    
    # Cevap içermeyen belirsiz ifadeler
    belirsiz_kaliplar = [
        'belirlenemez', 'bilgi yok', 'bilgi verilmedi',
        'yetersiz bilgi', 'daha fazla bilgi', 'hesaplanamaz',
        'mümkün değil', 'tanımsız', 'bilinmiyor',
        'tutarsız', 'kontrol edin', 'gözden geçir'
    ]
    # Türkçe yazılı sayılar
    yazili_sayilar = [
        'sıfır','bir','iki','üç','dört','beş',
        'altı','yedi','sekiz','dokuz','on','yirmi',
        'otuz','kırk','elli','altmış','yetmiş',
        'seksen','doksan','yüz','bin'
    ]
    
    for k in belirsiz_kaliplar:
        if k in son: return 'belirsiz'
    if re.findall(r'-?\d+(?:[.,]\d+)*', son):
        return 'rakam'
    for y in yazili_sayilar:
        if y in son: return 'yazili'
    return 'sayisal_olmayan'

# Tüm veri setini tara
sayac = Counter(sayi_tipi_tespit(ds['train'][i]['answer'])
                for i in range(len(ds['train'])))


In [ ]:
# Türkçe sayı formatında nokta binlik ayraç (11.000), virgül ondalık ayraç (0,5) olarak kullanılmakta.
# Bu durum sayı dönüşümünde özel işlem gerektirmektedir.
# Örnekler incelenerek is_correct fonksiyonuna doğru dönüşüm mantığı eklenmektedir.

nokta_ornekleri  = []
virgul_ornekleri = []

for i in range(len(ds['train'])):
    son = ds['train'][i]['answer'].strip().split('\n')[-1]
    if re.search(r'\d+\.\d{3}', son) and len(nokta_ornekleri) < 3:
        nokta_ornekleri.append((i, son))
    if re.search(r'\d+,\d+', son) and len(virgul_ornekleri) < 3:
        virgul_ornekleri.append((i, son))

print("Noktalı sayı örnekleri (binlik ayraç → 11.000 = 11000):")
for idx, s in nokta_ornekleri:
    print(f"  [{idx}] {s}")

print("\nVirgüllü sayı örnekleri (ondalık → 0,5 = 0.5):")
for idx, s in virgul_ornekleri:
    print(f"  [{idx}] {s}")

In [ ]:
# Türkçe sayı formatını doğru şekilde float'a dönüştüren fonksiyon buradda sayının tek nokta (1.000) bazen iki nokta 
# içerme durumu var buna göre bir tespit ve buna uygun dönüşümler yapılıyor.

def noktalı_sayi_isle(s):
    # virgüller kaldırılıyor.
    s = s.replace(',', '.')

    # nokta sayısına göre işlem yapılacak
    n = s.count('.')

    if n == 0:
        return float(s)
    elif n == 1:
        sol, sag = s.split('.')
        if sol == '0':
            # 0.9144 → ondalık sayı
            return float(s)
        elif len(sag) == 3:
            # 11.000 → binlik ayraç, noktayı kaldır
            return float(sol + sag)
        else:
            # 10.5 → ondalık sayı
            return float(s)
    elif n == 2:
        # 1.000.000 → tüm noktaları kaldır
        return float(s.replace('.', ''))
    return None
 
# Cevabın son satırından sayısal değeri çeken fonksiyon.
# Belirsiz veya çözümsüz cevaplar için None döndürür.
def son_sayi_cek(metin):
    son = metin.strip().split('\n')[-1]
    # Cevap içermeyen ya da belirsiz olan ifadeler None olarak işaretlenmekte.
    for k in ['belirlenemez','bilgi yok','bilgi verilmedi','yetersiz bilgi',
              'daha fazla bilgi','hesaplanamaz','mümkün değil','tanımsız',
              'bilinmiyor','tutarsız','kontrol edin','gözden geçir']:
        if k in son.lower(): return None
    sayilar = re.findall(r'-?\d+(?:[.,]\d+)*', son)
    return noktalı_sayi_isle(sayilar[-1]) if sayilar else None

# Modelin ürettiği cevabı ground truth ile karşılaştıran fonksiyon.
def is_correct(uretilen, gercek):
    p, t = son_sayi_cek(uretilen), son_sayi_cek(gercek)
    if p is None or t is None: return False
    return abs(p - t)


In [ ]:
# Bu test kodunu Ai ile yaptım 
# is_correct fonksiyonunun farklı sayı formatlarında doğru çalıştığını doğrulamak için test senaryoları.
testler = [
    ("Sonuç 216 kilogramdır.",
     "üretimini artırdıktan sonra bir yıl içinde 216 kilogram üzüme ihtiyacı olacaktır.", True),
    ("Sonuç 349 kuştur.",
     "Bir hafta sonra kümeste 349 kuş kalacaktır.", True),
    ("Sonuç 11.000 dolardır.",
     "James'in cebinden 11.000 $ çıkmıştır.", True),
    ("Betty'nin boyu 0.9144 metredir.",
     "Betty'nin boyu 0.9144 metredir.", True),
    ("Toplam 15 kuştur.",
     "Bir hafta sonra kümeste 349 kuş kalacaktır.", False),
    ("bilgi verilmediği için belirlenemez.",
     "Bir hafta sonra kümeste 349 kuş kalacaktır.", False),
    ("Simon alışverişinden 23,00$ bozuk para geri alır.",
     "Simon alışverişinden 23,00$ bozuk para geri alır.", True),
    ("Ian bir gül sakladı.",
     "Ian bir gül sakladı.", False),
]

print("TEST SONUÇLARI")
print("=" * 60)
tumu_gecti = True
for uretilen, gercek, beklenen in testler:
    sonuc  = is_correct(uretilen, gercek)
    durum  = "✅ GEÇTİ" if sonuc == beklenen else "❌ BAŞARISIZ"
    if sonuc != beklenen: tumu_gecti = False
    print(f"{durum} | Beklenen:{beklenen} Sonuç:{sonuc}")
    print(f"  → {uretilen[:70]}")
print("=" * 60)
print("Tüm testler geçti ✅" if tumu_gecti else "Bazı testler başarısız ❌")

In [ ]:
# Tarama için Qwen3.5-4B modeli yüklenmekte.
# model.eval() ile model çıkarım moduna alınmaktadır 
model_name = "Qwen/Qwen3.5-4B"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.bfloat16, device_map="auto"
)
model.eval()

In [ ]:
# Soru havuzundaki soruların taranması için gereken fonksiyonlar
# Tarama aşaması için logit toplamayan hızlı cevap üretim fonksiyonu.
# Logit toplamak bellek ve zaman maliyeti oluşturduğundan tarama aşamasında kullanılmamaktadır.
# Tarama sonrası seçilen sorular için answer_making.ipynb'de detaylı logit toplanmaktadır.
def uret_hizli(soru, max_new_tokens=200, temperature=1.0):
    messages = [{"role": "user", "content": soru}]
    # Modelin beklediği chat formatına dönüştürme.
    # enable_thinking=False: thinking mode kapalı tutulmakta — daha hızlı ve kısa cevaplar üretilmekte.
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False,
        add_generation_prompt=True, enable_thinking=False
    )
    inputs    = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]
    
    # torch.no_grad(): gradyan hesaplaması kapatılarak bellek ve hız optimize edilmektedir.
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, top_p=0.95, top_k=20,
            min_p=0.0, do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    # Prompt kısmı çıkarılarak sadece model çıktısı döndürülmekte.
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

# Fonksiyonun doğru çalıştığını test et
test_cevap = uret_hizli("35 + 2 kaç eder?")
print("Test cevabı:", test_cevap)

In [ ]:
# Belirsiz soruların tespiti için tarama döngüsü. Burada yapılıyor. 
# Burada belirsiz soruları belirlerken 1 soru için 3 kez cevap ürettiriyorum
# n az 1 doğru ve en az 1 yanlış varsa "aday" olarak belirliyorum daha sonrasında bu adaylara 10 kez cevap ürettireceğim.

# taranan dosyaların kayıt yeri
TARAMA_KAYIT = '/content/drive/MyDrive/proje5/data/tarama_adaylar.json'
HEDEF        = 500   # kaç belirsiz soru istiyoruz

# Kopmalar olduğu için tarama aşamasında Ai ile aşağıdaki yöntemi oluşturdum.
# Kaldığı yerden devam mekanizması
if os.path.exists(TARAMA_KAYIT):
    with open(TARAMA_KAYIT, encoding='utf-8') as f:
        kayit    = json.load(f)
    adaylar   = kayit['adaylar']
    baslangic = kayit['son_idx'] + 1
    print(f"⚡ Devam: {len(adaylar)} aday, idx={baslangic}'den başlıyor")
else:
    adaylar   = []
    baslangic = 0
    print("🆕 Sıfırdan başlıyor")

taranan = 0

for idx in range(baslangic, len(ds['train'])):
    soru   = ds['train'][idx]['question']
    gercek = ds['train'][idx]['answer']
    taranan += 1

    # Her soruya 3 cevap üretilmekte
    cevaplar     = [uret_hizli(soru) for _ in range(3)]
    dogru_sayisi = sum(is_correct(c, gercek) for c in cevaplar)

    # En az 1 doğru ve en az 1 yanlış cevap varsa belirsiz soru olarak işaretlenmekte
    if 0 < dogru_sayisi < 3:
        adaylar.append({
            "idx"         : idx,
            "soru"        : soru,
            "gercek_cevap": gercek,
            "on_dogru"    : dogru_sayisi
        })

    # Her 5 soruda bir ara kayıt yapılmakta bunu da Ai ile yaptım.
    if taranan % 5 == 0:
        with open(TARAMA_KAYIT, 'w', encoding='utf-8') as f:
            json.dump({"adaylar": adaylar, "son_idx": idx},
                      f, ensure_ascii=False, indent=2)

    if len(adaylar) >= HEDEF:
        print(f"{HEDEF} aday bulundu! ({taranan} soru tarandı)")
        break

# Final kayıt
with open(TARAMA_KAYIT, 'w', encoding='utf-8') as f:
    json.dump({"adaylar": adaylar, "son_idx": idx},
              f, ensure_ascii=False, indent=2)
    
print(f"Aday sayısı : {len(adaylar)}")
print(f"Dosya       : {TARAMA_KAYIT}")

In [ ]:
# Şimdi buradan sonra bütün veriler taranacak.
model_name = "Qwen/Qwen3.5B-4B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")
model.eval()  # Modeli değerlendirme moduna al

# Bu tarama işleminde drive kayıt edilecek belirsiz sorular.
# Burada belirsiz soru derken model aynı soruya bazen doğru cevap ve bazense yanlış cevap ürettiği sorular.
# Biz bu soruların cevaplarını üretilirken tokenlarını inceleyerek modelin neden belirsiz cevap ürettiğini anlamaya çalışacağız.

# Burada hızlı ilerlemek adına soruları belirlerken şöyle bir yöntem kullandım.
# İlk önce sadece 3 kez sorular soruldu bu soruların cevaplarından en az 1 doğru veya en az 1 yanlış cevap üreten sorular belirsiz olarak işaretlendi.
# Daha sonrasında bu işaretlenen sorulara 10 kez sorular sorulacak.

# Driveda uygun yere adaylar kayıt edilecek. 
TARAMA_KAYIT = '/content/drive/MyDrive/proje5/data/tarama_adaylar.json'
HEDEF        = 500   # kaç belirsiz soru istiyoruz

# Bunu Ai ile yaptım 
# Kaldığı yerden devam mekanizması
if os.path.exists(TARAMA_KAYIT):
    with open(TARAMA_KAYIT, encoding='utf-8') as f:
        kayit    = json.load(f)
    adaylar   = kayit['adaylar']
    baslangic = kayit['son_idx'] + 1
    print(f"⚡ Devam: {len(adaylar)} aday, idx={baslangic}'den başlıyor")
else:
    adaylar   = []
    baslangic = 0
    print("Sıfırdan başlıyor")


taranan = 0 # Ai önerisi ile taranan soruların sayısını tutmak için bir sayaç ekledim.

for idx in range(baslangic, len(ds['train'])):
    soru = ds['train'][idx]['question']
    gercek = ds['train'][idx]['answer']

    taranan += 1 # Ai önerisi ile taranan soruların sayısını tutmak için bir sayaç ekledim.

    # her soruya 3 cevap
    cevaplar = [uret_hizli(soru) for _ in range(3)]
    dogru_sayisi = sum(is_correct(c, gercek) for c in cevaplar)

    # En az 1 doğru veya en az 1 yanlış varsa belirsiz soru havuzuna ekleniyor.
    if 0 < dogru_sayisi < 3:
        adaylar.append({
            'idx': idx,
            'soru': soru,
            'gercek_cevap': gercek,
            'dogru_sayisi': dogru_sayisi
        })
    
    # Yaşadığım kopuntulardan dolayı Ai önerisi ile ağağıdaki bloğu ekledim
    # Her 5 soruda bir ara kayıt yapılmakta
    if taranan % 5 == 0:
        with open(TARAMA_KAYIT, 'w', encoding='utf-8') as f:
            json.dump({"adaylar": adaylar, "son_idx": idx},
                      f, ensure_ascii=False, indent=2)
    
    # hedefe ulaşınca döngüyü kır
    if len(adaylar) >= HEDEF:
        print(f"{HEDEF} aday bulundu! ({taranan} soru tarandı)")
        break


# Final kayıt
with open(TARAMA_KAYIT, 'w', encoding='utf-8') as f:
    json.dump({"adaylar": adaylar, "son_idx": idx},
              f, ensure_ascii=False, indent=2)
    
print(f"Aday sayısı : {len(adaylar)}")
print(f"Dosya       : {TARAMA_KAYIT}")